# Fase 6.4 — Despliegue en Produccion

**Pregunta de negocio:** Como ejecutar este analisis todos los dias en produccion?

**Objetivo:** Disenar e implementar una arquitectura de produccion que automatice
la ejecucion diaria del pipeline de analisis, el serving de modelos y el monitoreo
de operaciones usando servicios de Google Cloud Platform.

**Dataset:** `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`

---

## 0. Configuracion del Entorno

In [ ]:
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

import pandas as pd
import json
from IPython.display import display, HTML, Markdown

print("Entorno configurado correctamente.")

---
## 1. Arquitectura de Produccion

### 1.1 Diagrama General

```
                        ARQUITECTURA DE PRODUCCIÓN
                        ========================

  +-----------------+     +-------------------+     +------------------+
  | Cloud Scheduler |---->| Cloud Functions   |---->| BigQuery         |
  | (Cron 6AM UTC)  |     | (Python 3.11)     |     | (Consultas SQL)  |
  +-----------------+     +-------------------+     +------------------+
                                  |                        |
                                  |                        v
                                  |                 +------------------+
                                  |                 | BigQuery Tables  |
                                  |                 | (Agregados)      |
                                  |                 +------------------+
                                  |                        |
                                  v                        v
                          +------------------+     +------------------+
                          | Cloud Storage    |     | Looker Studio    |
                          | (Parquet/CSV)    |     | (Dashboards)     |
                          +------------------+     +------------------+
                                  |
                                  v
                          +------------------+     +------------------+
                          | Vertex AI        |---->| API Endpoint     |
                          | (Model Serving)  |     | (Predicciones)   |
                          +------------------+     +------------------+
                                                          |
                                                          v
                          +------------------+     +------------------+
                          | Cloud Monitoring |     | Alertas          |
                          | (Logs/Metricas)  |---->| (Email/Slack)    |
                          +------------------+     +------------------+
```

### 1.2 Flujo de Datos

1. **Cloud Scheduler** dispara la ejecucion diaria a las 6:00 AM UTC
2. **Cloud Functions** ejecuta las consultas de agregacion en BigQuery
3. **BigQuery** procesa los datos y escribe tablas de agregados
4. **Cloud Storage** almacena resultados intermedios en formato parquet
5. **Vertex AI** carga los datos nuevos, ejecuta inferencia con modelos entrenados
6. **Looker Studio** se conecta a BigQuery para actualizar dashboards
7. **Cloud Monitoring** detecta fallos y envia alertas

---
## 2. Cloud Scheduler: Programacion de Ejecucion

Cloud Scheduler permite crear jobs tipo cron que envian una solicitud HTTP a un endpoint
(en nuestro caso, la URL de una Cloud Function).

### 2.1 Configuracion del Job

In [ ]:
# Configuración de Cloud Scheduler via gcloud CLI
scheduler_config = """
# ====================================================================
# Crear job de Cloud Scheduler
# ====================================================================

# Variables de configuración
PROJECT_ID="mi-proyecto-gcp"
REGION="us-central1"
FUNCTION_URL="https://${REGION}-${PROJECT_ID}.cloudfunctions.net/nyc-taxi-daily-pipeline"
SERVICE_ACCOUNT="nyc-taxi-scheduler@${PROJECT_ID}.iam.gserviceaccount.com"

# Crear el job
gcloud scheduler jobs create http nyc-taxi-daily-pipeline \\
    --location=${REGION} \\
    --schedule="0 6 * * *" \\
    --uri=${FUNCTION_URL} \\
    --http-method=POST \\
    --headers="Content-Type=application/json" \\
    --message-body='{"date": "today", "run_inference": true}' \\
    --oidc-service-account-email=${SERVICE_ACCOUNT} \\
    --time-zone="America/New_York" \\
    --description="Pipeline diario de análisis NYC Taxi" \\
    --attempt-deadline=600s

# Verificar el job
gcloud scheduler jobs describe nyc-taxi-daily-pipeline --location=${REGION}

# Ejecutar manualmente para probar
gcloud scheduler jobs run nyc-taxi-daily-pipeline --location=${REGION}
"""

print(scheduler_config)

### 2.2 Expresion Cron

```
0 6 * * *
| | | | |
| | | | +-- Dia de la semana (0-7, * = todos)
| | | +---- Mes (1-12, * = todos)
| | +------ Dia del mes (1-31, * = todos)
| +-------- Hora (0-23)
+---------- Minuto (0-59)
```

**Significado:** Ejecutar todos los dias a las 6:00 AM (hora de Nueva York).

**Justificacion:** Las 6 AM permite que los agregados esten listos antes de la jornada
laboral (que comienza tipicamente a las 8-9 AM). Tambien permite capturar todos los
viajes del dia anterior (hasta las ~4 AM cuando baja la actividad nocturna).

---
## 3. Cloud Function: Codigo del Pipeline Diario

La Cloud Function es el corazon del pipeline. Ejecuta las consultas de agregacion,
guarda resultados y opcionalmente ejecuta inferencia con los modelos.

In [ ]:
# Código de la Cloud Function: main.py
cloud_function_code = '''
import functions_framework
from google.cloud import bigquery
from google.cloud import storage
import pandas as pd
import joblib
import json
import logging
from datetime import datetime, timedelta
import io

# Configuración
PROJECT_ID = "mi-proyecto-gcp"
DATASET_ID = "nyc_taxi_analytics"
BUCKET_NAME = "nyc-taxi-pipeline-results"
MODEL_BUCKET = "nyc-taxi-models"

# Zonas TLC por borough
MANHATTAN_ZONES = "\'4\',\'12\',\'13\',\'24\',\'41\',\'42\',\'43\',\'45\',\'48\',\'50\',\'68\',\'74\',\'75\',\'79\',\'87\',\'88\',\'90\',\'100\',\'107\',\'113\',\'114\',\'116\',\'125\',\'127\',\'128\',\'137\',\'140\',\'141\',\'142\',\'143\',\'144\',\'148\',\'151\',\'152\',\'153\',\'158\',\'161\',\'162\',\'163\',\'164\',\'166\',\'170\',\'186\',\'194\',\'202\',\'209\',\'211\',\'224\',\'229\',\'230\',\'231\',\'232\',\'233\',\'234\',\'236\',\'237\',\'238\',\'239\',\'243\',\'244\',\'246\',\'249\',\'261\',\'262\',\'263\'"
BROOKLYN_ZONES = "\'11\',\'14\',\'17\',\'21\',\'22\',\'25\',\'26\',\'29\',\'33\',\'34\',\'35\',\'36\',\'37\',\'39\',\'40\',\'49\',\'52\',\'54\',\'55\',\'61\',\'62\',\'63\',\'65\',\'66\',\'67\',\'69\',\'71\',\'72\',\'76\',\'77\',\'80\',\'85\',\'89\',\'91\',\'97\',\'106\',\'108\',\'111\',\'112\',\'123\',\'133\',\'149\',\'150\',\'154\',\'155\',\'165\',\'177\',\'178\',\'181\',\'188\',\'189\',\'190\',\'195\',\'210\',\'217\',\'222\',\'225\',\'227\',\'228\',\'255\',\'256\',\'257\'"

# Clientes
bq_client = bigquery.Client(project=PROJECT_ID)
storage_client = storage.Client(project=PROJECT_ID)

logger = logging.getLogger(__name__)


@functions_framework.http
def nyc_taxi_daily_pipeline(request):
    """Pipeline diario de análisis de NYC Taxi."""
    try:
        # Parsear parámetros
        request_json = request.get_json(silent=True) or {}
        run_inference = request_json.get("run_inference", True)

        # Determinar fecha de análisis (ayer)
        target_date = (datetime.utcnow() - timedelta(days=1)).strftime("%Y-%m-%d")
        logger.info(f"Ejecutando pipeline para fecha: {target_date}")

        # Paso 1: Ejecutar consulta de agregación diaria
        daily_stats = run_daily_aggregation(target_date)
        logger.info(f"Agregación diaria completada: {len(daily_stats)} registros")

        # Paso 2: Guardar resultados en Cloud Storage
        save_to_gcs(daily_stats, target_date)
        logger.info("Resultados guardados en Cloud Storage")

        # Paso 3: Ejecutar inferencia con modelos (opcional)
        if run_inference:
            predictions = run_model_inference(daily_stats)
            logger.info(f"Inferencia completada: {len(predictions)} predicciones")

        return json.dumps({
            "status": "success",
            "date": target_date,
            "records_processed": len(daily_stats),
            "inference_run": run_inference
        }), 200

    except Exception as e:
        logger.error(f"Error en pipeline: {str(e)}")
        return json.dumps({"status": "error", "message": str(e)}), 500


def run_daily_aggregation(target_date):
    """Ejecuta la consulta de agregación diaria en BigQuery."""
    query = f"""
    SELECT
        DATE(pickup_datetime) AS trip_date,
        EXTRACT(HOUR FROM pickup_datetime) AS hour,
        CASE
            WHEN pickup_location_id IN ({MANHATTAN_ZONES}) THEN \'Manhattan\'
            WHEN pickup_location_id IN ({BROOKLYN_ZONES}) THEN \'Brooklyn\'
            WHEN pickup_location_id = \'132\' THEN \'JFK\'
            WHEN pickup_location_id = \'138\' THEN \'LaGuardia\'
            ELSE \'Queens/Otros\'
        END AS zone,
        COUNT(*) AS trips,
        AVG(fare_amount) AS avg_fare,
        AVG(tip_amount) AS avg_tip,
        AVG(trip_distance) AS avg_distance,
        SUM(fare_amount + tip_amount) AS total_revenue
    FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
    WHERE DATE(pickup_datetime) = \'{target_date}\'
      AND fare_amount BETWEEN 2.5 AND 500
      AND trip_distance BETWEEN 0.1 AND 100
      AND pickup_location_id IS NOT NULL
    GROUP BY trip_date, hour, zone
    ORDER BY hour, zone
    """

    df = bq_client.query(query).to_dataframe()
    return df


def save_to_gcs(df, target_date):
    """Guarda DataFrame como parquet en Cloud Storage."""
    bucket = storage_client.bucket(BUCKET_NAME)
    blob_name = f"daily_aggregates/{target_date}.parquet"
    blob = bucket.blob(blob_name)

    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False)
    buffer.seek(0)
    blob.upload_from_file(buffer, content_type="application/octet-stream")


def run_model_inference(daily_stats):
    """Carga modelos desde GCS y ejecuta predicciones."""
    # Descargar modelo de tarifa
    bucket = storage_client.bucket(MODEL_BUCKET)
    blob = bucket.blob("fare_model.joblib")
    model_bytes = blob.download_as_bytes()
    fare_model = joblib.load(io.BytesIO(model_bytes))

    # Preparar features para predicción
    features = daily_stats[["hour", "avg_distance"]].copy()
    features["zone_encoded"] = daily_stats["zone"].astype("category").cat.codes

    # Ejecutar predicción
    daily_stats["predicted_fare"] = fare_model.predict(features)

    return daily_stats
'''

print("=" * 70)
print("CLOUD FUNCTION: main.py")
print("=" * 70)
print(cloud_function_code)

In [ ]:
# requirements.txt para la Cloud Function
requirements_cf = """
# requirements.txt para Cloud Function
functions-framework==3.*
google-cloud-bigquery>=3.10.0
google-cloud-storage>=2.10.0
pandas>=2.0.0
pyarrow>=12.0.0
joblib>=1.3.0
scikit-learn>=1.3.0
xgboost>=2.0.0
"""

print("requirements.txt para Cloud Function:")
print(requirements_cf)

In [ ]:
# Comando para desplegar la Cloud Function
deploy_commands = """
# ====================================================================
# Desplegar Cloud Function
# ====================================================================

PROJECT_ID="mi-proyecto-gcp"
REGION="us-central1"
SERVICE_ACCOUNT="nyc-taxi-pipeline@${PROJECT_ID}.iam.gserviceaccount.com"

# Desplegar la función
gcloud functions deploy nyc-taxi-daily-pipeline \\
    --gen2 \\
    --runtime=python311 \\
    --region=${REGION} \\
    --source=. \\
    --entry-point=nyc_taxi_daily_pipeline \\
    --trigger-http \\
    --allow-unauthenticated=false \\
    --service-account=${SERVICE_ACCOUNT} \\
    --memory=512MiB \\
    --timeout=300s \\
    --max-instances=1 \\
    --set-env-vars="PROJECT_ID=${PROJECT_ID}"

# Verificar despliegue
gcloud functions describe nyc-taxi-daily-pipeline --region=${REGION}
"""

print(deploy_commands)

---
## 4. BigQuery: Consultas Programadas y Vistas Materializadas

Ademas de la Cloud Function, podemos usar las consultas programadas nativas de BigQuery
para mantener tablas de agregados actualizadas automaticamente.

In [ ]:
# Consulta programada: tabla de agregados diarios
scheduled_query_daily = """
-- ====================================================================
-- Consulta Programada: Agregados Diarios
-- Frecuencia: Diaria a las 6:30 AM UTC
-- Destino: nyc_taxi_analytics.daily_aggregates
-- ====================================================================

CREATE OR REPLACE TABLE `mi-proyecto-gcp.nyc_taxi_analytics.daily_aggregates` AS
SELECT
    DATE(pickup_datetime) AS trip_date,
    COUNT(*) AS total_trips,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(CASE WHEN payment_type = '1' AND fare_amount > 0
              THEN tip_amount / fare_amount END) * 100, 1) AS avg_tip_pct,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, SECOND)) / 60.0, 1) AS avg_duration_min,
    ROUND(SUM(fare_amount), 2) AS total_fare_revenue,
    ROUND(SUM(tip_amount), 2) AS total_tip_revenue,
    ROUND(SUM(fare_amount + tip_amount), 2) AS total_revenue,
    COUNTIF(payment_type = '1') AS credit_card_trips,
    COUNTIF(payment_type = '2') AS cash_trips
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE fare_amount BETWEEN 2.5 AND 500
  AND trip_distance BETWEEN 0.1 AND 100
  AND pickup_datetime >= '2015-01-01'
  AND pickup_datetime < '2016-01-01'
GROUP BY trip_date
ORDER BY trip_date;
"""

print(scheduled_query_daily)

In [ ]:
# Consulta programada: tabla de agregados por hora y zona
scheduled_query_hourly = """
-- ====================================================================
-- Consulta Programada: Agregados por Hora y Zona
-- Frecuencia: Diaria a las 6:45 AM UTC
-- Destino: nyc_taxi_analytics.hourly_zone_aggregates
-- ====================================================================

CREATE OR REPLACE TABLE `mi-proyecto-gcp.nyc_taxi_analytics.hourly_zone_aggregates` AS
SELECT
    DATE(pickup_datetime) AS trip_date,
    EXTRACT(HOUR FROM pickup_datetime) AS hour,
    EXTRACT(DAYOFWEEK FROM pickup_datetime) AS day_of_week,
    CASE
        WHEN pickup_location_id IN ('4','12','13','24','41','42','43','45','48','50','68','74','75','79','87','88','90','100','107','113','114','116','125','127','128','137','140','141','142','143','144','148','151','152','153','158','161','162','163','164','166','170','186','194','202','209','211','224','229','230','231','232','233','234','236','237','238','239','243','244','246','249','261','262','263')
        THEN 'Manhattan'
        WHEN pickup_location_id IN ('11','14','17','21','22','25','26','29','33','34','35','36','37','39','40','49','52','54','55','61','62','63','65','66','67','69','71','72','76','77','80','85','89','91','97','106','108','111','112','123','133','149','150','154','155','165','177','178','181','188','189','190','195','210','217','222','225','227','228','255','256','257')
        THEN 'Brooklyn'
        WHEN pickup_location_id = '132' THEN 'JFK'
        WHEN pickup_location_id = '138' THEN 'LaGuardia'
        ELSE 'Queens/Otros'
    END AS zone,
    COUNT(*) AS trips,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(tip_amount), 2) AS avg_tip,
    ROUND(AVG(trip_distance), 2) AS avg_distance
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE fare_amount BETWEEN 2.5 AND 500
  AND trip_distance BETWEEN 0.1 AND 100
  AND pickup_location_id IS NOT NULL
  AND pickup_datetime >= '2015-01-01'
  AND pickup_datetime < '2016-01-01'
GROUP BY trip_date, hour, day_of_week, zone
ORDER BY trip_date, hour, zone;
"""

print(scheduled_query_hourly)

### 4.1 Vistas Materializadas

Las vistas materializadas son tablas pre-calculadas que BigQuery mantiene automaticamente
sincronizadas con la tabla base. Son ideales para consultas frecuentes y repetitivas.

**Ventajas:**
- Se actualizan automaticamente cuando los datos base cambian
- Las consultas que acceden a la vista materializada no procesan la tabla completa
- Reduccion significativa de costos para consultas repetitivas

**Limitaciones:**
- Solo soportan un subconjunto de funciones SQL
- No soportan JOINs complejos
- Requieren que la tabla base este en el mismo proyecto

In [ ]:
# Ejemplo de vista materializada
materialized_view_sql = """
-- ====================================================================
-- Vista Materializada: Resumen por Zona
-- Se actualiza automáticamente cuando cambian los datos base
-- ====================================================================

CREATE MATERIALIZED VIEW IF NOT EXISTS
  `mi-proyecto-gcp.nyc_taxi_analytics.mv_zone_summary`
AS
SELECT
    CASE
        WHEN pickup_location_id IN ('4','12','13','24','41','42','43','45','48','50','68','74','75','79','87','88','90','100','107','113','114','116','125','127','128','137','140','141','142','143','144','148','151','152','153','158','161','162','163','164','166','170','186','194','202','209','211','224','229','230','231','232','233','234','236','237','238','239','243','244','246','249','261','262','263')
        THEN 'Manhattan'
        WHEN pickup_location_id IN ('11','14','17','21','22','25','26','29','33','34','35','36','37','39','40','49','52','54','55','61','62','63','65','66','67','69','71','72','76','77','80','85','89','91','97','106','108','111','112','123','133','149','150','154','155','165','177','178','181','188','189','190','195','210','217','222','225','227','228','255','256','257')
        THEN 'Brooklyn'
        WHEN pickup_location_id = '132' THEN 'JFK'
        WHEN pickup_location_id = '138' THEN 'LaGuardia'
        ELSE 'Queens/Otros'
    END AS zone,
    COUNT(*) AS total_trips,
    AVG(fare_amount) AS avg_fare,
    AVG(tip_amount) AS avg_tip,
    AVG(trip_distance) AS avg_distance
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE fare_amount BETWEEN 2.5 AND 500
  AND pickup_location_id IS NOT NULL
GROUP BY zone;

-- Nota: Para datos propios (no públicos), esto reduciría el costo
-- de consultas repetitivas a prácticamente $0.
"""

print(materialized_view_sql)

---
## 5. Vertex AI: Despliegue de Modelos

Vertex AI permite servir modelos de ML como endpoints REST para predicciones en tiempo real.

### 5.1 Subir Artefacto del Modelo

In [ ]:
# Paso 1: Subir modelo a Cloud Storage
upload_model_commands = """
# ====================================================================
# Subir artefactos del modelo a Cloud Storage
# ====================================================================

PROJECT_ID="mi-proyecto-gcp"
BUCKET="gs://nyc-taxi-models"

# Crear bucket (si no existe)
gsutil mb -p ${PROJECT_ID} -l us-central1 ${BUCKET}

# Subir modelos
gsutil cp models/nyc_taxi/fare_model.joblib ${BUCKET}/fare_model/model.joblib
gsutil cp models/nyc_taxi/tip_model.joblib ${BUCKET}/tip_model/model.joblib
gsutil cp models/nyc_taxi/high_tip_classifier.joblib ${BUCKET}/high_tip_classifier/model.joblib

# Verificar
gsutil ls ${BUCKET}/
"""

print(upload_model_commands)

### 5.2 Crear Endpoint en Vertex AI

In [ ]:
# Código para registrar modelo y crear endpoint en Vertex AI
vertex_ai_code = '''
from google.cloud import aiplatform

# Inicializar Vertex AI
PROJECT_ID = "mi-proyecto-gcp"
REGION = "us-central1"
BUCKET = "gs://nyc-taxi-models"

aiplatform.init(project=PROJECT_ID, location=REGION)

# ====================================================================
# Paso 1: Registrar el modelo
# ====================================================================
model = aiplatform.Model.upload(
    display_name="nyc-taxi-fare-predictor",
    artifact_uri=f"{BUCKET}/fare_model/",
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest",
    description="XGBoost model para predicción de tarifas de taxi NYC. R²=0.95",
    labels={"project": "nyc-taxi", "phase": "production", "model_type": "regression"}
)

print(f"Modelo registrado: {model.display_name}")
print(f"Resource name: {model.resource_name}")

# ====================================================================
# Paso 2: Crear endpoint
# ====================================================================
endpoint = aiplatform.Endpoint.create(
    display_name="nyc-taxi-fare-endpoint",
    description="Endpoint para predicción en tiempo real de tarifas",
    labels={"project": "nyc-taxi"}
)

print(f"Endpoint creado: {endpoint.display_name}")
print(f"Resource name: {endpoint.resource_name}")

# ====================================================================
# Paso 3: Desplegar modelo en endpoint
# ====================================================================
model.deploy(
    endpoint=endpoint,
    deployed_model_display_name="fare-predictor-v1",
    machine_type="n1-standard-2",  # 2 vCPUs, 7.5 GB RAM
    min_replica_count=1,
    max_replica_count=3,  # Autoescalado
    traffic_percentage=100
)

print("Modelo desplegado exitosamente.")
'''

print(vertex_ai_code)

### 5.3 Prediccion en Linea (Ejemplo)

In [ ]:
# Ejemplo de predicción online
prediction_example = '''
from google.cloud import aiplatform

# Conectar al endpoint existente
endpoint = aiplatform.Endpoint(
    endpoint_name="projects/mi-proyecto-gcp/locations/us-central1/endpoints/ENDPOINT_ID"
)

# Datos de entrada para predicción
# Cada instancia es un viaje con sus features
instances = [
    {
        "trip_distance": 3.5,       # millas
        "duration_min": 15.0,       # minutos
        "hour": 17,                 # 5 PM (hora pico)
        "day_of_week": 3,           # miércoles
        "zone_pickup": 0,           # Manhattan (encoded)
        "passenger_count": 1
    },
    {
        "trip_distance": 18.0,      # viaje al aeropuerto
        "duration_min": 45.0,
        "hour": 8,                  # 8 AM
        "day_of_week": 1,           # lunes
        "zone_pickup": 0,           # Manhattan
        "passenger_count": 2
    }
]

# Ejecutar predicción
predictions = endpoint.predict(instances=instances)

print("Predicciones de tarifa:")
for i, pred in enumerate(predictions.predictions):
    print(f"  Viaje {i+1}: ${pred:.2f}")

# Resultado esperado:
# Viaje 1: ~$14.50 (viaje típico Manhattan)
# Viaje 2: ~$52.00 (viaje al aeropuerto JFK)
'''

print(prediction_example)

---
## 6. Monitoreo y Alertas

Un sistema de produccion necesita monitoreo proactivo para detectar fallos
antes de que impacten a los usuarios.

### 6.1 Cloud Monitoring para Cloud Functions

Google Cloud automaticamente registra metricas de ejecucion de Cloud Functions:

| Metrica | Descripcion | Alerta Sugerida |
|---------|-------------|------------------|
| `execution_count` | Numero de ejecuciones | Si < 1/dia, alerta |
| `execution_times` | Duracion de ejecucion | Si > 240s, alerta |
| `memory_usage` | Uso de memoria | Si > 80% del limite |
| `error_count` | Numero de errores | Si > 0, alerta |
| `status` | Codigo HTTP de respuesta | Si != 200, alerta |

In [ ]:
# Configuración de alertas con gcloud
monitoring_config = """
# ====================================================================
# Configurar Alertas de Cloud Monitoring
# ====================================================================

# 1. Alerta por fallo de Cloud Function
# Se puede configurar via consola o con Terraform/gcloud

# Política de alerta: error en Cloud Function
cat > alert_policy_function_error.json << 'POLICY'
{
  "displayName": "NYC Taxi Pipeline - Error en Cloud Function",
  "conditions": [
    {
      "displayName": "Error count > 0",
      "conditionThreshold": {
        "filter": "resource.type=\"cloud_function\" AND resource.labels.function_name=\"nyc-taxi-daily-pipeline\" AND metric.type=\"cloudfunctions.googleapis.com/function/execution_count\" AND metric.labels.status!=\"ok\"",
        "comparison": "COMPARISON_GT",
        "thresholdValue": 0,
        "duration": "0s",
        "aggregations": [
          {
            "alignmentPeriod": "300s",
            "perSeriesAligner": "ALIGN_SUM"
          }
        ]
      }
    }
  ],
  "notificationChannels": ["projects/mi-proyecto-gcp/notificationChannels/CHANNEL_ID"],
  "alertStrategy": {
    "autoClose": "604800s"
  }
}
POLICY

gcloud alpha monitoring policies create --policy-from-file=alert_policy_function_error.json

# 2. Configurar canal de notificación por email
gcloud beta monitoring channels create \\
    --display-name="Equipo Data Science" \\
    --type=email \\
    --channel-labels=email_address=equipo-ds@empresa.com

# 3. Configurar canal de Slack (requiere webhook URL)
gcloud beta monitoring channels create \\
    --display-name="Canal Slack #alertas-pipeline" \\
    --type=slack \\
    --channel-labels=channel_name="#alertas-pipeline"
"""

print(monitoring_config)

### 6.2 BigQuery Audit Logs

BigQuery registra automaticamente todas las consultas ejecutadas. Esto es util para:
- Rastrear el consumo de bytes procesados
- Identificar consultas costosas
- Auditar accesos a los datos

```sql
-- Consultar logs de auditoría de BigQuery (últimas 24 horas)
SELECT
    timestamp,
    protopayload_auditlog.methodName,
    protopayload_auditlog.servicedata_v1_bigquery.jobCompletedEvent.job.jobStatistics.totalBytesProcessed,
    protopayload_auditlog.servicedata_v1_bigquery.jobCompletedEvent.job.jobStatistics.totalSlotMs,
    protopayload_auditlog.authenticationInfo.principalEmail
FROM `mi-proyecto-gcp.logs.cloudaudit_googleapis_com_data_access`
WHERE timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR)
  AND protopayload_auditlog.methodName = 'jobservice.jobcompleted'
ORDER BY timestamp DESC
LIMIT 50;
```

### 6.3 Alertas de Anomalia

Podemos integrar la deteccion de anomalias (Isolation Forest) en el pipeline diario
para enviar alertas cuando un dia se desvie significativamente del patron esperado.

In [ ]:
# Ejemplo: función de detección de anomalías para alertas
anomaly_alert_code = '''
import smtplib
from email.mime.text import MIMEText
from sklearn.ensemble import IsolationForest
import numpy as np


def check_for_anomalies(today_stats, historical_stats):
    """
    Verifica si las estadísticas de hoy son anómalas
    comparadas con el histórico.

    Args:
        today_stats: dict con {trips, avg_fare, avg_tip_pct}
        historical_stats: DataFrame con columnas [trips, avg_fare, avg_tip_pct]

    Returns:
        bool: True si es anómalo
    """
    # Entrenar Isolation Forest con datos históricos
    iso_forest = IsolationForest(contamination=0.05, random_state=42)
    iso_forest.fit(historical_stats[["trips", "avg_fare", "avg_tip_pct"]])

    # Predecir si hoy es anómalo
    today_array = np.array([[today_stats["trips"],
                             today_stats["avg_fare"],
                             today_stats["avg_tip_pct"]]])

    prediction = iso_forest.predict(today_array)
    is_anomaly = prediction[0] == -1

    if is_anomaly:
        send_anomaly_alert(today_stats)

    return is_anomaly


def send_anomaly_alert(stats):
    """Envía alerta por email cuando se detecta una anomalía."""
    subject = f"[ALERTA] Anomalía detectada en NYC Taxi Pipeline"
    body = f"""
    Se ha detectado una anomalía en las estadísticas de hoy:

    - Viajes: {stats['trips']:,.0f}
    - Tarifa promedio: ${stats['avg_fare']:.2f}
    - Propina promedio: {stats['avg_tip_pct']:.1f}%

    Por favor, investigue la causa (posible evento climático,
    festivo, o problema de datos).

    -- Pipeline Automático NYC Taxi
    """

    # En producción, usar SendGrid, Cloud Tasks, o Pub/Sub
    print(f"ALERTA: {subject}")
    print(body)
'''

print(anomaly_alert_code)

---
## 7. Estimacion de Costos

Desglose detallado de los costos mensuales para ejecutar este pipeline en produccion.

In [ ]:
# Tabla de estimación de costos
costs = pd.DataFrame({
    'Servicio': [
        'BigQuery (consultas diarias)',
        'BigQuery (almacenamiento tablas)',
        'Cloud Functions (ejecucion diaria)',
        'Cloud Storage (resultados)',
        'Cloud Scheduler',
        'Vertex AI (endpoint 24/7)',
        'Vertex AI (predicciones)',
        'Cloud Monitoring',
        'Cloud Logging'
    ],
    'Uso Estimado': [
        '~40 GB/dia procesados',
        '~1 GB almacenamiento activo',
        '1 ejecucion/dia, ~60s, 512MB',
        '~50 MB/mes de parquets',
        '1 job',
        'n1-standard-2, 24/7',
        '~1,000 predicciones/dia',
        'Metricas basicas',
        '<50 GB/mes'
    ],
    'Costo_Diario_USD': [
        0.20, 0.001, 0.01, 0.001, 0.003, 3.50, 0.05, 0.00, 0.00
    ],
    'Costo_Mensual_USD': [
        6.00, 0.02, 0.30, 0.02, 0.10, 105.00, 1.50, 0.00, 0.00
    ]
})

print("=" * 80)
print("ESTIMACION DE COSTOS MENSUALES - NYC Taxi Pipeline")
print("=" * 80)
display(costs)

total_daily = costs['Costo_Diario_USD'].sum()
total_monthly = costs['Costo_Mensual_USD'].sum()

print(f"\nTotal diario estimado:  ${total_daily:.2f}")
print(f"Total mensual estimado: ${total_monthly:.2f}")
print(f"Total anual estimado:   ${total_monthly * 12:.2f}")

### 7.1 Opciones para Reducir Costos

El mayor costo es **Vertex AI** (~$105/mes por un endpoint 24/7). Alternativas:

| Opcion | Descripcion | Costo Estimado |
|--------|-------------|----------------|
| **Endpoint 24/7** | Predicciones en tiempo real | ~$105/mes |
| **Batch prediction** | Predicciones por lote 1x/dia | ~$2-5/mes |
| **Cloud Run** | Contenedor que escala a 0 | ~$5-15/mes |
| **Sin endpoint** | Solo predicciones en Cloud Function | ~$0.30/mes |

**Recomendacion para este proyecto:** Usar **batch prediction** o integrar la inferencia
directamente en la Cloud Function, lo que reduce el costo total a **~$5-12/mes**.

### 7.2 Desglose Optimizado

| Servicio | Costo/mes (Optimizado) |
|----------|------------------------|
| BigQuery | ~$6.00 |
| Cloud Functions | ~$0.30 |
| Cloud Storage | ~$0.02 |
| Cloud Scheduler | ~$0.10 |
| **Total Optimizado** | **~$6.50/mes** |

---
## 8. Consideraciones de Escalado

### 8.1 Escalado Horizontal

| Componente | Estrategia de Escalado |
|------------|------------------------|
| **BigQuery** | Escala automaticamente. No requiere configuracion adicional. |
| **Cloud Functions** | Escala automaticamente hasta `max_instances`. |
| **Vertex AI** | Configurar `min/max_replica_count` para autoescalado. |
| **Cloud Storage** | Escala automaticamente. Sin limites practicos. |

### 8.2 Escalado del Volumen de Datos

| Escenario | Datos/dia | Adaptaciones Necesarias |
|-----------|-----------|-------------------------|
| Actual (2015) | ~400K viajes | Ninguna |
| 2x volumen | ~800K viajes | Aumentar timeout de CF a 600s |
| 10x volumen | ~4M viajes | Migrar a Cloud Dataflow/Dataproc |
| Streaming | Tiempo real | Usar Pub/Sub + Dataflow streaming |

### 8.3 Escalado de Modelos

Para multiples modelos en produccion, considerar:

1. **Model Registry:** Usar Vertex AI Model Registry para versionar modelos
2. **A/B Testing:** Desplegar multiples versiones en el mismo endpoint con traffic split
3. **Feature Store:** Usar Vertex AI Feature Store para features compartidas
4. **MLOps Pipeline:** Implementar CI/CD para modelos con Vertex AI Pipelines

---
## 9. Seguridad

### 9.1 Service Accounts e IAM

Cada componente del pipeline debe usar un service account dedicado con permisos minimos
(principio de menor privilegio).

In [ ]:
# Configuración de service accounts y roles IAM
security_config = """
# ====================================================================
# Configuración de Seguridad
# ====================================================================

PROJECT_ID="mi-proyecto-gcp"

# 1. Crear service account para el pipeline
gcloud iam service-accounts create nyc-taxi-pipeline \\
    --display-name="NYC Taxi Pipeline Service Account" \\
    --description="Service account para el pipeline diario de NYC Taxi"

SA_EMAIL="nyc-taxi-pipeline@${PROJECT_ID}.iam.gserviceaccount.com"

# 2. Asignar roles mínimos necesarios

# BigQuery: solo lectura en dataset público + escritura en dataset propio
gcloud projects add-iam-policy-binding ${PROJECT_ID} \\
    --member="serviceAccount:${SA_EMAIL}" \\
    --role="roles/bigquery.dataEditor" \\
    --condition='expression=resource.name.startsWith("projects/${PROJECT_ID}/datasets/nyc_taxi_analytics"),title=nyc-taxi-dataset-only'

gcloud projects add-iam-policy-binding ${PROJECT_ID} \\
    --member="serviceAccount:${SA_EMAIL}" \\
    --role="roles/bigquery.jobUser"

# Cloud Storage: lectura/escritura en buckets específicos
gsutil iam ch serviceAccount:${SA_EMAIL}:objectCreator,objectViewer \\
    gs://nyc-taxi-pipeline-results

gsutil iam ch serviceAccount:${SA_EMAIL}:objectViewer \\
    gs://nyc-taxi-models

# Vertex AI: solo predicción (no puede crear/eliminar modelos)
gcloud projects add-iam-policy-binding ${PROJECT_ID} \\
    --member="serviceAccount:${SA_EMAIL}" \\
    --role="roles/aiplatform.user"

# 3. Crear service account separado para Cloud Scheduler
gcloud iam service-accounts create nyc-taxi-scheduler \\
    --display-name="NYC Taxi Scheduler Service Account"

SCHEDULER_SA="nyc-taxi-scheduler@${PROJECT_ID}.iam.gserviceaccount.com"

# Solo puede invocar la Cloud Function específica
gcloud functions add-invoker-policy-binding nyc-taxi-daily-pipeline \\
    --region=us-central1 \\
    --member="serviceAccount:${SCHEDULER_SA}"
"""

print(security_config)

### 9.2 Resumen de Roles por Service Account

| Service Account | Roles | Justificacion |
|----------------|-------|---------------|
| `nyc-taxi-pipeline@` | BigQuery Data Editor (dataset propio), BigQuery Job User, Storage Object Creator/Viewer, AI Platform User | Necesita leer/escribir datos y ejecutar modelos |
| `nyc-taxi-scheduler@` | Cloud Functions Invoker | Solo puede disparar la Cloud Function |

### 9.3 Proteccion de Datos

- **No almacenar coordenadas exactas en tablas propias:** Usar solo agregados por zona/cuadricula.
- **Encriptacion:** BigQuery y Cloud Storage encriptan datos en reposo por defecto (AES-256).
- **VPC Service Controls (opcional):** Para entornos regulados, configurar un perimetro de servicio
  que restrinja la salida de datos.
- **Data Loss Prevention (DLP):** No aplica directamente (datos publicos), pero en produccion
  con datos propios, usar Cloud DLP para detectar informacion sensible.
- **Audit Logs:** Mantener habilitados los logs de acceso a datos en BigQuery y Cloud Storage.

---
## Conclusion

Este notebook documenta una arquitectura de produccion completa para el pipeline
de analisis NYC Taxi, cubriendo:

1. **Arquitectura** con diagrama de flujo de datos
2. **Cloud Scheduler** para ejecucion diaria automatizada
3. **Cloud Function** con codigo Python completo para el pipeline
4. **BigQuery** con consultas programadas y vistas materializadas
5. **Vertex AI** para despliegue y serving de modelos
6. **Monitoreo** con Cloud Monitoring, alertas y deteccion de anomalias
7. **Costos** desglosados (~$5-12/mes optimizado, ~$113/mes con endpoint 24/7)
8. **Escalado** con estrategias para crecimiento de datos
9. **Seguridad** con service accounts, IAM y proteccion de datos

El pipeline puede desplegarse en aproximadamente 1-2 horas siguiendo los comandos
y codigo proporcionados. La configuracion optimizada cuesta menos de $10/mes.

---
*Notebook creado como parte de la Fase 6 (Sintesis y Produccion) del proyecto NYC Taxi.*